## Prep

In [1]:
import json
import sys
import pandas as pd
import collections 
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["CUDA_LAUNCH_BLOCKING"] = "3"
import numpy as np
from itertools import chain
from itertools import combinations
sys.path.insert(0, '..')
from src.experiment_utils.helper_classes import token, span, repository
from src.d02_corpus_statistics.corpus import Corpus
import types
from owlready2 import sync_reasoner
from collections import Counter
from transformers import pipeline, AutoModel, PreTrainedTokenizerBase
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer
import spacy
from spacy.training import offsets_to_biluo_tags
from datasets import Dataset, DatasetDict
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
import evaluate
from typing import Any, Dict, List
from sklearn.metrics import f1_score

cwd = os.getcwd()
pol_dir = cwd+"/../src/d01_data"

/home/marwas/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


so many things to try even just for NER, especially given how long some of these token sequences are

trying multiple classifiction heads
- different base models
- using last hidden state vs using last few hidden states (average or concatenation)
- weighted vs unweighted loss
- evaluation metrics (token micro F1 or overlap instead of seqeval)

#### Create Dataset

In [2]:
pol_df = pd.read_pickle(pol_dir+"/preprocessed_dataframe.pkl")[["Policy","Text","Tokens","Curation"]]

https://medium.com/@shahrukhx01/multi-task-learning-with-transformers-part-1-multi-prediction-heads-b7001cf014bf

In [3]:
def span_to_bio_tok_lbls(feature_name, tokens, spans, label2id):
    token_labels = ["O"] * len(tokens)
    for spn in spans:
        if spn.feature == feature_name:
            start_char = spn.start
            end_char = spn.stop
            inside_tokens = []
            for i, tok in enumerate(tokens):
                tok_start = tok.start
                tok_end = tok.stop
                overlap = not (tok_end <= start_char or tok_start >= end_char)
                if overlap:
                    inside_tokens.append(i)
            if inside_tokens:
                token_labels[inside_tokens[0]] = f"B"
                for i in inside_tokens[1:]:
                    token_labels[i] = f"I"
    return [label2id[l] for l in token_labels]

def df_to_dataset(df):
    label2id = {
        "O":0, "B":1, "I":2
    }
    dataset = {
        "id":[],
        "text":[],
        "tokens":[],
        "labels_Actor":[],
        "labels_InstrumentType":[],
        "labels_Objective":[],
        "labels_Resource":[],
        "labels_Time":[]
    }
    for artid in df.index:
        tokens = df.loc[artid,"Tokens"]
        if len(tokens) <= 512: # we'll change this eventually
            text = df.loc[artid,"Text"]
            spans = df.loc[artid,"Curation"]
            token_texts = [t.text for t in tokens]
            dataset['id'].append(artid)
            dataset["text"].append(text)
            dataset["tokens"].append(token_texts)
            for ftr in ["Actor", "InstrumentType", "Objective", "Resource", "Time"]:
                token_level_labels = span_to_bio_tok_lbls(ftr, tokens, spans, label2id)
                dataset[f"labels_{ftr}"].append(token_level_labels)
    return Dataset.from_dict(dataset), list(label2id)

In [4]:
dataset, label_list = df_to_dataset(pol_df)
id2label = {}
label2id = {}
for i, lbl in enumerate(label_list):
    id2label[i] = lbl
    label2id[lbl] = i
# do the datasets need to differ by model used for tokenization of results too??

In [ ]:
'''
for r in [0,1,2]:
    td_test = dataset.train_test_split(test_size=0.2, seed=r)
    train_dev = td_test['train'].train_test_split(test_size=0.25, seed=r)
    ds_dct = DatasetDict({"train":train_dev['train'], "dev":train_dev['test'], "test":td_test['test']})
    print(ds_dct)
    ds_dct.save_to_disk(cwd+f"/inputs/sep/dsdct_r{r}")
'''

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 190
    })
    dev: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 64
    })
    test: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 64
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 64/64 [00:00<00:00, 10187.69 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 190
    })
    dev: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 64
    })
    test: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 64
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 64/64 [00:00<00:00, 12580.75 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 190
    })
    dev: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 64
    })
    test: Dataset({
        features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
        num_rows: 64
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 64/64 [00:00<00:00, 12093.87 examples/s]


''

## Tokenize

In [2]:
label2id = {
        "O":0, "B":1, "I":2
    }
id2label = {
    0:"O", 1:"B", 2:"I"
}

In [3]:
model_name = "microsoft/deberta-v3-base" # suggested lr of 3e-5
#model_name = "dslim/bert-base-NER-uncased"
#model_name = "FacebookAI/xlm-roberta-base"

have to adapt the tokenizing and aligning script to account for the separate label lists

In [31]:
# all feature/label types
label_cols = [
    "labels_Actor",
    "labels_InstrumentType",
    "labels_Objective",
    "labels_Resource",
    "labels_Time"
]

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    # adapted for multi-head from https://huggingface.co/docs/transformers/en/tasks/token_classification
    # even tho the token lists area already split into words, we need to break them into subwords
    # and then ensure that the label sequences still align in the new token sequence
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, padding=True, return_attention_mask=True)
    # for each label type/list
    for col in label_cols:
        all_aligned_labels = []
        # loop through this label type's sequence in each sample and realign
        for sample_idx, labels in enumerate(examples[col]):
            word_ids = tokenized_inputs.word_ids(batch_index=sample_idx)
            # smth like [None, 0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 19, 20]
            previous_word_idx = None
            label_ids = []
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(labels[word_idx])
                else:
                    label_ids.append(-100)
                previous_word_idx = word_idx
            all_aligned_labels.append(label_ids)
        tokenized_inputs[col] = all_aligned_labels
    return tokenized_inputs

/home/marwas/.local/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [32]:
r=0
dataset_dict = DatasetDict.load_from_disk(cwd+f"/inputs/sep/dsdct_r{r}")
tokenized_dsdct = dataset_dict.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/190 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Map: 100%|██████████| 64/64 [00:00<00:00, 717.29 examples/s]


In [23]:
# sanity checkign
tokenized_inputs = tokenizer(dataset_dict['train'][0:5]["tokens"], truncation=True, is_split_into_words=True)
for sample_idx, labels in enumerate(dataset_dict['train'][0:5]['labels_Actor']):
    word_ids = tokenized_inputs.word_ids(batch_index=sample_idx)
    print(word_ids)

[None, 0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 29, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, None]
[None, 0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 

new custom data collator for multi-heads

we need a new data collator because we have mutliple label lists

In [52]:
class MultiHeadDataCollator:
    '''
    Using PreTrainedTokenizerBase i.e. whatever pretrained tokenizer we have from tokenize_and_align_labels
    And using pad_sequence
    '''
    def __init__(self, tokenizer: PreTrainedTokenizerBase, label_columns: List[str], padding=True, max_length=None):
        #initializing the essentials
        self.tokenizer = tokenizer
        self.label_columns = label_columns
        self.padding = padding
        self.max_length = max_length
    def __call__(self, features):
        input_ids = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
        #attention_mask = [torch.tensor(f["attention_mask"], dtype=torch.long) for f in features] # something isnt working
        attention_mask = [
            torch.tensor(f.get("attention_mask", [1]*len(f["input_ids"])), dtype=torch.long)
            for f in features
        ]
        # padding inputids and attnmask
        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
        # batch
        batch = {
            "input_ids": input_ids,
            "attention_mask": attention_mask
        }
        # padding labels
        for col in self.label_columns:
            #label_lists = [torch.tensor(f[col], dtype=torch.long) for f in features] # something isnt working here either
            label_lists = [
                torch.tensor(f.get(col, [-100]*len(f["input_ids"])), dtype=torch.long)
                for f in features
            ]
            labels_padded = pad_sequence(label_lists, batch_first=True, padding_value=-100)
            # then finally adding to batch
            batch[col] = labels_padded
        return batch

In [48]:
# sanity checking
data_collator = MultiHeadDataCollator(tokenizer=tokenizer, label_columns=label_cols, max_length=512)
sample_batch = [tokenized_dsdct['train'][i] for i in range(16)]
batch = data_collator(sample_batch)
for k, v in batch.items():
    print(k, v.shape)

input_ids torch.Size([16, 507])
attention_mask torch.Size([16, 507])
labels_Actor torch.Size([16, 507])
labels_InstrumentType torch.Size([16, 507])
labels_Objective torch.Size([16, 507])
labels_Resource torch.Size([16, 507])
labels_Time torch.Size([16, 507])


In [38]:
#sanity checking
tokens = tokenizer.convert_ids_to_tokens(batch["input_ids"][0])
for idx, tok in enumerate(tokens[:50]):
    print(f"{idx:03} | {tok:15} | "
          f"A:{batch['labels_Actor'][0][idx].item():2}  "
          f"T:{batch['labels_Time'][0][idx].item():2}  "
          f"I:{batch['labels_InstrumentType'][0][idx].item():2}")


000 | [CLS]           | A:-100  T:-100  I:-100
001 | ▁article        | A: 0  T: 0  I: 0
002 | ▁18             | A: 0  T: 0  I: 0
003 | ▁bills          | A: 0  T: 0  I: 0
004 | ▁and            | A: 0  T: 0  I: 0
005 | ▁billing        | A: 0  T: 0  I: 0
006 | ▁information    | A: 0  T: 0  I: 0
007 | ▁1              | A: 0  T: 0  I: 0
008 | ▁.              | A: 0  T: 0  I: 0
009 | ▁member         | A: 1  T: 0  I: 0
010 | ▁states         | A: 2  T: 0  I: 0
011 | ▁shall          | A: 0  T: 0  I: 0
012 | ▁ensure         | A: 0  T: 0  I: 0
013 | ▁that           | A: 0  T: 0  I: 0
014 | ▁bills          | A: 0  T: 0  I: 0
015 | ▁and            | A: 0  T: 0  I: 0
016 | ▁billing        | A: 0  T: 0  I: 0
017 | ▁information    | A: 0  T: 0  I: 0
018 | ▁are            | A: 0  T: 0  I: 0
019 | ▁accurate       | A: 0  T: 0  I: 0
020 | ▁,              | A: 0  T: 0  I: 0
021 | ▁easy           | A: 0  T: 0  I: 0
022 | ▁to             | A: 0  T: 0  I: 0
023 | ▁understand     | A: 0  T: 0  I: 0
024 | ▁,  

new model structure

microsoft/deberta-v3-base

In [10]:
# lets create class (BIO) weights for each feature type
num_classes = 3
class_weights = {}
for lname in label_cols:
    name = lname.replace("labels_", "")
    labels = np.concatenate([
        np.array(l) for l in tokenized_dsdct['train'][lname]
    ])
    labels = labels[labels != -100]  # remove padding
    counter = Counter(labels)
    total = sum(counter.values())
    class_weights[name] = torch.tensor([total / counter[i] for i in range(num_classes)], dtype=torch.float)
class_weights

{'Actor': tensor([ 1.0680, 29.4342, 33.6430]),
 'InstrumentType': tensor([ 1.0448, 49.2159, 44.3662]),
 'Objective': tensor([  1.0540, 160.6564,  22.2236]),
 'Resource': tensor([  1.0100, 278.3893, 159.2533]),
 'Time': tensor([  1.0236, 179.6503,  57.0720])}

In [11]:
# how many labeled individual tokens, how many labeled spans
head_counts_all = {}
head_counts_ents = {}
for lname in label_cols:
    name = lname.replace("labels_", "")
    # concatenate all labels and remove -100s
    all_labels = np.concatenate([np.array(l) for l in tokenized_dsdct['train'][lname]])
    all_labels = all_labels[all_labels != -100]
    all_labels = all_labels[all_labels != 0]
    head_counts_all[name] = len(all_labels) # only tokens B or I
    #all_labels = all_labels[all_labels != 2]
    #head_counts_ents[name] = len(all_labels) # only B tokens 
print("All:", head_counts_all)
#print("Ents:", head_counts_ents)

total_tokens_all = sum(head_counts_all.values())
head_weights_all = {head: total_tokens_all / head_counts_all[head] for head in head_counts_all}
min_weight_all = min(head_weights_all.values())
head_weights_min_all = {k: v / min_weight_all for k, v in head_weights_all.items()}
print("\nAll")
print("Normal:", head_weights_all)
print("Min:", head_weights_min_all)
'''
total_tokens_ents = sum(head_counts_ents.values())
head_weights_ents = {head: total_tokens_ents / head_counts_ents[head] for head in head_counts_ents}
min_weight_ents = min(head_weights_ents.values())
head_weights_min_ents = {k: v / min_weight_ents for k, v in head_weights_ents.items()}
print("\nEnts")
print("Normal:", head_weights_ents)
print("Min:", head_weights_min_ents)
'''

All: {'Actor': 2323, 'InstrumentType': 1563, 'Objective': 1868, 'Resource': 360, 'Time': 842}

All
Normal: {'Actor': 2.9944037882049073, 'InstrumentType': 4.450415866922585, 'Objective': 3.7237687366167025, 'Resource': 19.322222222222223, 'Time': 8.261282660332542}
Min: {'Actor': 1.0, 'InstrumentType': 1.486244401791427, 'Objective': 1.2435760171306212, 'Resource': 6.452777777777778, 'Time': 2.7589073634204277}


'\ntotal_tokens_ents = sum(head_counts_ents.values())\nhead_weights_ents = {head: total_tokens_ents / head_counts_ents[head] for head in head_counts_ents}\nmin_weight_ents = min(head_weights_ents.values())\nhead_weights_min_ents = {k: v / min_weight_ents for k, v in head_weights_ents.items()}\nprint("\nEnts")\nprint("Normal:", head_weights_ents)\nprint("Min:", head_weights_min_ents)\n'

In [12]:
head_weights = {
    'Actor': 1.0, 
    'InstrumentType': 1.486244401791427, 
    'Objective': 1.2435760171306212, 
    'Resource': 6.452777777777778, 
    'Time': 2.7589073634204277
}

since loss is computed per token instead of per span, we'll look at the All instead of the Ents results

In [56]:
class DebertaForMultiHeadTokClass(nn.Module):
    def __init__(self):
        super().__init__()
        n_labels = 3
        # shared encoder
        self.base_model = AutoModel.from_pretrained('microsoft/deberta-v3-base')
        hidden_size = self.base_model.config.hidden_size
        #sep linear head for each feature type classification
        self.classifiers = nn.ModuleDict({
            "Actor": nn.Linear(hidden_size, n_labels),
            "InstrumentType": nn.Linear(hidden_size, n_labels),
            "Objective": nn.Linear(hidden_size, n_labels),
            "Resource": nn.Linear(hidden_size, n_labels),
            "Time": nn.Linear(hidden_size, n_labels)
        })
    def forward(self, input_ids, attention_mask=None, **labels):
        # batch of inputs encoded by base model
        outputs = self.base_model(input_ids, attention_mask=attention_mask)
        # only uses last hidden state... for now
        # will look into averaging/concatenating last few hidden states
        sequence_output = outputs.last_hidden_state
        # passes encoded input sequence to each classifier to get logits
        logits = {name: self.classifiers[name](sequence_output) for name in self.classifiers}
        loss = 0
        if labels:
            # for labels_Feature, tensor(batch_sz,seq_ln)
            for lname, label in labels.items():
                if label is not None:
                    name = lname.replace("labels_", "")
                    # flatten attn mask
                    active_loss = attention_mask.view(-1) == 1
                    # get active logits, flatten to (num_act_tokens, num_classes) 
                    # then apply active loss mask (to both logits and labels)
                    active_logits = logits[name].view(-1, 3)[active_loss]
                    active_labels = label.view(-1)[active_loss]
                    # weighting BIO classes for this feature
                    weight = class_weights[name].to(active_logits.device)
                    loss_fct = nn.CrossEntropyLoss(weight=weight)
                    # computing loss for this head
                    head_loss = loss_fct(active_logits, active_labels)
                    # weight the loss for this head
                    head_loss *= head_weights[name]
                    # sum loss across heads for single update to train simultaneously
                    loss += head_loss
        # for multi-head classification we'll return like this
        return {"loss": loss, "logits": logits}

In [ ]:
# work on model that concatenates or averages last few hidden states for encoded representation

In [57]:
model = DebertaForMultiHeadTokClass()

new compute_metrics for token micro-f1 instead of seqeval

In [ ]:
def compute_metrics_multihead(p):
    prediction_dct, label_dct = p
    metrics = {}
    # for each head
    for head_name, logits in prediction_dct.items():
        labels = label_dct[head_name] # size (batch, seq_len)
        labels_flat = labels.flatten()
        preds_flat = np.argmax(logits, axis=-1).flatten()
        # mask out -100s
        mask = labels_flat != -100
        labels_flat = labels_flat[mask]
        preds_flat = preds_flat[mask]
        # micro F1
        f1 = f1_score(labels_flat, preds_flat, average='micro')
        metrics[f"{head_name}_f1"] = f1

In [53]:
data_collator = MultiHeadDataCollator(tokenizer=tokenizer, label_columns=label_cols, max_length=512)

In [62]:
training_args = TrainingArguments(
    output_dir=model_name.split("/")[-1],
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    #load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dsdct["train"],
    eval_dataset=tokenized_dsdct["dev"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_multihead
)

add early stopping to trainer?

In [ ]:
trainer.train()
trainer.save_model(cwd+f"/models/sep/{model_name.split('/')[-1]}_{r}")
del model
del trainer

Epoch,Training Loss,Validation Loss
1,No log,No log
2,No log,No log
3,No log,No log
4,No log,No log


## Test

old metrics (seqeval)

In [ ]:
seqeval = evaluate.load("seqeval")
results_dict = {
    "microsoft/deberta-v3-base":{},
    "FacebookAI/xlm-roberta-base":{},
    "dslim/bert-base-NER-uncased":{}
}
for model_name in list(results_dict):
    results_dict[model_name]["Overall"] = {"precision":[], "recall":[], "f1":[], "accuracy":[]}
    for ftr in ["Actor", "InstrumentType", "Objective", "Resource", "Time"]:
        results_dict[model_name][ftr] = {"precision":[], "recall":[], "f1":[], "number":[]}
for model_name in list(results_dict):
    for r in [0,1,2]:
        dataset_dict = DatasetDict.load_from_disk(cwd+f"/inputs/all/dsdct_r{r}")
        model_tt = AutoModelForTokenClassification.from_pretrained(cwd+f"/models/all/{model_name.split('/')[-1]}_{r}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        classifier = pipeline("ner", model = model_tt, tokenizer=tokenizer)
        inputs = tokenizer(list(dataset_dict['test']['text']), return_tensors="pt", padding=True, truncation=True)
        device = torch.device("cuda")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = model_tt(**inputs).logits
        predictions = torch.argmax(logits, dim=2)
        seqeval = evaluate.load("seqeval")
        labels = list(dataset_dict['test']['ner_tags'])
        true_predictions = [
            [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]
        true_labels = [
            [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]
        results = seqeval.compute(predictions=true_predictions, references=true_labels)
        for k in list(results):
            if k[:4]=="over":
                x, metric = k.split("_")
                results_dict[model_name]['Overall'][metric].append(float(results[k]))
            else:
                for mtr in list(results[k]):
                    results_dict[model_name][k][mtr].append(float(results[k][mtr]))           

In [ ]:
for m in list(results_dict):
    print(f"\n{m}")
    for res in list(results_dict[m]):
        print(f"{res}")
        print(results_dict[m][res])

In [ ]:
fn = "2nd_results"
with open(cwd+f"/outputs/{fn}.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, indent=4)

In [ ]:
for m in list(results_dict):
    print(f"\n{m}")
    for res in list(results_dict[m]):
        print(f"\n{res}")
        df = pd.DataFrame(results_dict[m][res])
        df.loc['mean'] = df.mean()
        print(df)

new metrics -- token micro f1